<a href="https://colab.research.google.com/github/amaimanwar8-arch/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/amaimanwar8-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Working dir:", os.getcwd())
print(df.shape[0], "pages loaded successfully.")

Working dir: /content/flyrank-ml-internship
30000 pages loaded successfully.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [2]:
print("""
Method: Random Forest classifier.

I have real prior evidence for this choice, not just intuition: in Notebook 2's readable-
tree exercise, a random forest reached Precision@50 = 0.740 versus the hand-written
baseline's 0.240 - roughly 3x more of the top 50 flagged pages were actually correct, on
this same task and label. Random forest also fits Lane 2's own reasoning from ML-03: the
signals here (position, freshness, CTR, word count) interact in ways a short if/else rule
can't capture past 2-3 conditions, but a tree ensemble can weigh many of them together
without needing careful feature scaling.
""")


Method: Random Forest classifier.

I have real prior evidence for this choice, not just intuition: in Notebook 2's readable-
tree exercise, a random forest reached Precision@50 = 0.740 versus the hand-written
baseline's 0.240 - roughly 3x more of the top 50 flagged pages were actually correct, on
this same task and label. Random forest also fits Lane 2's own reasoning from ML-03: the
signals here (position, freshness, CTR, word count) interact in ways a short if/else rule
can't capture past 2-3 conditions, but a tree ensemble can weigh many of them together
without needing careful feature scaling.



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
from sklearn.model_selection import GroupShuffleSplit

visible = df[df["impressions_90d"] >= 100].copy()
model_features = ["content_age_days", "days_since_last_update", "impressions_90d",
                   "avg_position", "ctr", "word_count"]
visible[model_features] = visible[model_features].replace([np.inf, -np.inf], np.nan).fillna(0)

X = visible[model_features]
y = visible["is_declining_label"]
groups = visible["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
visible_te = visible.iloc[test_idx].copy()

train_clients = set(visible.iloc[train_idx]["client_id"])
test_clients = set(visible_te["client_id"])

print(f"Train: {len(X_tr):,} rows from {len(train_clients)} clients")
print(f"Test: {len(X_te):,} rows from {len(test_clients)} clients")
print(f"Client overlap between train and test (must be 0): {len(train_clients & test_clients)}")

print("""
Split design: grouped by client_id, not random. Pages from the same client can share
writing style, template, and topical patterns - a random split could let the model
"memorize" a client seen in training and get an easy win on that same client's pages in
"test," which would overstate how well it generalizes to clients it's never seen. A
client-grouped split forces the model to prove itself on genuinely new clients, matching
the lane guide's own validation rule for this exact risk.
""")

Train: 17,396 rows from 22 clients
Test: 4,610 rows from 8 clients
Client overlap between train and test (must be 0): 0

Split design: grouped by client_id, not random. Pages from the same client can share
writing style, template, and topical patterns - a random split could let the model
"memorize" a client seen in training and get an easy win on that same client's pages in
"test," which would overstate how well it generalizes to clients it's never seen. A
client-grouped split forces the model to prove itself on genuinely new clients, matching
the lane guide's own validation rule for this exact risk.



## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# --- Baseline score, recomputed on the SAME test set (same logic as w04) ---
tier_avg_ctr_te = visible_te.groupby("position_tier")["ctr"].transform("mean")
ctr_gap_te = (tier_avg_ctr_te - visible_te["ctr"]).clip(lower=0)
ctr_gap_norm_te = ctr_gap_te / ctr_gap_te.max()
aging_flag_te = (visible_te["freshness_tier"] == "91-180").astype(int)
baseline_score_te = (0.65 * ctr_gap_norm_te) + (0.35 * aging_flag_te)

# --- Model, trained on train, scored on test ---
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
model_score_te = model.predict_proba(X_te)[:, 1]

results = []
for k in (20, 50):
    b = precision_at_k(baseline_score_te.values, y_te.values, k)
    m = precision_at_k(model_score_te, y_te.values, k)
    results.append({"k": k, "baseline_precision": round(b, 3), "model_precision": round(m, 3)})

results_df = pd.DataFrame(results)
print("Model vs. baseline, SAME client-grouped test set, SAME metric (Precision@K):")
results_df

Model vs. baseline, SAME client-grouped test set, SAME metric (Precision@K):


,k,baseline_precision,model_precision
0,20,0.85,0.65
1,50,0.84,0.76


In [5]:
print("""
Result: the baseline BEATS the model on this client-grouped test set.
Precision@20: baseline 0.850 vs. model 0.650
Precision@50: baseline 0.840 vs. model 0.760

This is a different story than Notebook 2, where the model won big (0.740 vs 0.240) - but
that comparison used a RANDOM split, not a client-grouped one. Notebook 2's model likely
partly "memorized" per-client patterns it later saw again in its own test rows. Once
genuinely new clients are held out, the model's advantage disappears and the simple
CTR-gap + aging rule actually generalizes better here.

Honest takeaway: my current 6-feature random forest does not beat the baseline under fair
validation. That's a real, useful finding, not a failed notebook - it means either (a) the
baseline's two signals already capture most of what's learnable with this feature set, or
(b) the model needs more/better features, more trees, or tuning to actually add value
beyond what a human already reasoned out from Signal 1 and Signal 2. I'm keeping the
baseline as the stronger choice for now rather than reporting an inflated model win.
""")


Result: the baseline BEATS the model on this client-grouped test set.
Precision@20: baseline 0.850 vs. model 0.650
Precision@50: baseline 0.840 vs. model 0.760

This is a different story than Notebook 2, where the model won big (0.740 vs 0.240) - but
that comparison used a RANDOM split, not a client-grouped one. Notebook 2's model likely
partly "memorized" per-client patterns it later saw again in its own test rows. Once
genuinely new clients are held out, the model's advantage disappears and the simple
CTR-gap + aging rule actually generalizes better here.

Honest takeaway: my current 6-feature random forest does not beat the baseline under fair
validation. That's a real, useful finding, not a failed notebook - it means either (a) the
baseline's two signals already capture most of what's learnable with this feature set, or
(b) the model needs more/better features, more trees, or tuning to actually add value
beyond what a human already reasoned out from Signal 1 and Signal 2. I'm keep

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
from sklearn.inspection import permutation_importance

importances = permutation_importance(model, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature": model_features,
    "importance": importances.importances_mean
}).sort_values("importance", ascending=False)
print("Permutation importance (what the model actually leans on):")
print(imp_df)

visible_te["model_score"] = model_score_te
visible_te["predicted_declining"] = (model_score_te >= 0.5).astype(int)
false_positives = visible_te[(visible_te["predicted_declining"] == 1) & (visible_te["is_declining_label"] == 0)]
false_negatives = visible_te[(visible_te["predicted_declining"] == 0) & (visible_te["is_declining_label"] == 1)]

print(f"\nFalse positives (model said declining, wasn't): {len(false_positives):,}")
print(f"False negatives (model said fine, was declining): {len(false_negatives):,}")

print(f"""
Actual interpretation:

Top feature by permutation importance is content_age_days (0.028), followed closely by
ctr (0.021) and avg_position (0.020) - these three carry most of the model's real signal.
Notably, days_since_last_update has a NEGATIVE importance (-0.015), meaning shuffling it
actually helped the model slightly - it's contributing noise, not signal, despite being
the exact feature my baseline's aging_flag was built from. This is a useful disagreement:
my baseline bet on freshness/aging mattering, but the model finds content_age_days (how
old the page is overall) more informative than days_since_last_update (how long since it
was touched) - two different notions of "age" that aren't behaving the same way.

Error volume: {len(false_positives):,} false positives and {len(false_negatives):,} false
negatives out of {len(X_te):,} test rows - meaningful disagreement on unseen clients, not
a small edge case. Combined with the baseline beating the model on Precision@K, this
suggests my current 6-feature set doesn't give the random forest enough to clearly beat a
well-reasoned 2-signal rule. A stronger next step would be testing content_age_days-based
signals more directly, rather than assuming days_since_last_update is the right aging
proxy - which is itself a finding this error analysis surfaced, not something I assumed
going in.
""")

Permutation importance (what the model actually leans on):
                  feature  importance
0        content_age_days    0.028134
4                     ctr    0.020911
3            avg_position    0.020499
2         impressions_90d    0.017570
5              word_count    0.007852
1  days_since_last_update   -0.014989

False positives (model said declining, wasn't): 1,061
False negatives (model said fine, was declining): 841

Actual interpretation:

Top feature by permutation importance is content_age_days (0.028), followed closely by
ctr (0.021) and avg_position (0.020) - these three carry most of the model's real signal.
Notably, days_since_last_update has a NEGATIVE importance (-0.015), meaning shuffling it
actually helped the model slightly - it's contributing noise, not signal, despite being
the exact feature my baseline's aging_flag was built from. This is a useful disagreement:
my baseline bet on freshness/aging mattering, but the model finds content_age_days (how
old the p

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.